# MTCG: Multi-Template Computational Graph for Traffic Demand Flow Estimation

**Melbourne Network — Path Visualization**

**Authors:**
- Xin (Bruce) Wu, Department of Civil and Environmental Engineering, Villanova University, USA
- Feng Shao, School of Mathematics, China University of Mining and Technology, China

**Contact:** xwu03@villanova.edu

---

MIT License

Copyright (c) 2026 Xin (Bruce) Wu, Feng Shao

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# Path Visualization — Melbourne Network

This notebook selects representative OD pairs and visualizes their template-specific routes on an interactive map.

In [1]:
import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap
from pyproj import Proj, transform

from pyproj import Transformer
from tqdm.notebook import trange

## 1. Data Loading

Load OD pairs, agent path data, and demand for all three templates.

In [2]:
od_pair = pd.read_csv('data_new_1/OD_pair.csv').values
num_od = len(od_pair)
od_pair

array([[1920, 1981],
       [1920, 1983],
       [1920, 1995],
       ...,
       [2348, 2345],
       [2348, 2346],
       [2348, 2347]], dtype=int64)

In [3]:
path1 = pd.read_csv('data_new_1/agent_new.csv')
path2 = pd.read_csv('data_new_2/agent_new.csv')
path3 = pd.read_csv('data_new_3/agent_new.csv')

In [4]:
demand_6_10 = pd.read_csv('data_new_1/demand_6-10.csv').values.T
demand_6_10.shape

(16, 30040)

In [5]:
# Timestep
T = 12
num_sample = 16 - T + 1
num_sample

5

In [6]:
demand_gt = np.zeros([num_sample, T, num_od])
for i in range(num_sample):
    demand_gt[i] = demand_6_10[i:i+T]
demand_gt.shape

(5, 12, 30040)

In [7]:
demand_reference = demand_gt[4:,:,:]
demand_reference.shape

(1, 12, 30040)

In [8]:
d1 = demand_reference[0,0,:]
d1.shape

(30040,)

In [9]:
index_d1 = np.argsort(d1)
index_d1

array([    0, 19604, 19603, ...,  1220,  4421,  1758], dtype=int64)

## 2. OD Pair Selection

Select high-demand OD pairs with diverse, non-looping paths across all templates.

In [10]:
# # 0: -3
# # 12: -2
# index1 = index_d1[-6]
# d1[index1]

import numpy as np

sorted_od_idx = np.argsort(d1)[::-1]

cnt1 = path1.groupby(['o_zone_id','d_zone_id']).size()
cnt2 = path2.groupby(['o_zone_id','d_zone_id']).size()
cnt3 = path3.groupby(['o_zone_id','d_zone_id']).size()

def get_3seq(df, o, d):
    s = df.loc[(df['o_zone_id']==o) & (df['d_zone_id']==d), 'node_sequence']
    return s[:3].reset_index(drop=True)

def loop_ratio(seq):
    ids = [int(x) for x in str(seq).split(';') if x!='']
    if len(ids) == 0:
        return 1.0
    return 1.0 - (len(set(ids)) / len(ids))

def mean_loop_ratio(seqs):
    return float(np.mean([loop_ratio(seqs.iloc[i]) for i in range(3)]))

TOP_K = 300
THR = 0.15

cands = []  # Store all qualifying candidates (score, demand, j)

for j in sorted_od_idx[:TOP_K]:
    o = od_pair[j,0]; d = od_pair[j,1]

    if (cnt1.get((o,d),0) < 3) or (cnt2.get((o,d),0) < 3) or (cnt3.get((o,d),0) < 3):
        continue

    seqs = get_3seq(path1, o, d)
    if len(seqs) < 3:
        continue

    score = mean_loop_ratio(seqs)
    if score > THR:
        continue

    cands.append((score, float(d1[j]), int(j)))

if len(cands) == 0:
    raise RuntimeError("No qualifying OD found in top-K: need 3/3/3 paths without loops. Increase TOP_K or relax thresholds.")

# Sort by lower score (more normal) first, then higher demand
cands.sort(key=lambda x: (x[0], -x[1]))

# ===== Force selection: pick N-th candidate (0-based) =====
N = 1   # 0=best, 1=2nd, 4=5th, etc.
N = min(N, len(cands)-1)

best_score, best_demand, index1 = cands[N]

print("Forced selection of", N+1, "candidate: index1 =", index1,
      "OD =", od_pair[index1,0], od_pair[index1,1],
      "demand =", d1[index1],
      "loop_score =", best_score,
      "total candidates =", len(cands))

强制选第 2 名候选：index1 = 1859 OD = 1940 2150 demand = 50.0 loop_score = 0.0 候选总数 = 86


In [11]:
p1 = path1.loc[(path1['o_zone_id']==od_pair[index1,0])&(path1['d_zone_id']==od_pair[index1,1]),'node_sequence']
p1 = p1[:3]
p1.reset_index(drop=True,inplace=True)

p2 = path2.loc[(path2['o_zone_id']==od_pair[index1,0])&(path2['d_zone_id']==od_pair[index1,1]),'node_sequence']
p2 = p2[:3]
p2.reset_index(drop=True,inplace=True)


p3 = path3.loc[(path3['o_zone_id']==od_pair[index1,0])&(path3['d_zone_id']==od_pair[index1,1]),'node_sequence']
p3 = p3[:3]
p3.reset_index(drop=True,inplace=True)

print(len(p1))
print(len(p2))
print(len(p3))

3
3
3


In [12]:
node_unique = pd.read_csv('data_new_1/node_unique.csv').values

node_unique = node_unique.reshape(-1)
node_unique.shape

(2348,)

In [13]:
node = pd.read_csv('data_new_1/node.csv')
node_xy = pd.read_csv('data_new_1/node_xy.csv')   # This table has valid x_coord/y_coord (not NaN)
node.shape

(2348, 11)

In [14]:
node

,node_id,zone_id,name,bin_index,node_type,control_type,x_coord,y_coord,geometry,production,attraction
0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
1,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
2,3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
3,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
4,5,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
...,...,...,...,...,...,...,...,...,...,...,...
2343,2344,2344,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
2344,2345,2345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
2345,2346,2346,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
2346,2347,2347,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1


## 3. Trajectory Generation

Convert node sequences to geographic coordinates (UTM → WGS84).

In [15]:
def sequence_to_trajectory(x):
    
    node_sequence = []
    
    list_x = x.split(';')
    int_x = [int(num) for num in list_x]
    array_x = np.array(int_x)
    
    # path_node = node_unique[array_x-1]
    path_node = array_x
    
    for n in range(len(path_node)):
        
        # x = node.loc[node['node_id']==path_node[n],'x_coord'].values
        # y = node.loc[node['node_id']==path_node[n],'y_coord'].values
        x = node_xy.loc[node_xy['node_id']==path_node[n], 'x_coord'].values
        y = node_xy.loc[node_xy['node_id']==path_node[n], 'y_coord'].values
        
        if len(x)>0:
            node_sequence.append((x[0],y[0]))
    
    return node_sequence

In [16]:
path_trajectory1 = sequence_to_trajectory(p3[0])
path_trajectory2 = sequence_to_trajectory(p3[1])
path_trajectory3 = sequence_to_trajectory(p3[2])

In [17]:
# Define WGS84 coordinate system (EPSG:4326)
wgs84 = Proj(init='epsg:4326')

# Define UTM zone for Melbourne area, e.g. 55S (EPSG:28355)
# Note: adjust the EPSG code based on the actual UTM zone used
utm_zone = Proj(init='epsg:32755')  # UTM zone 55S

D:\ProgramData\anaconda3\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
D:\ProgramData\anaconda3\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


In [18]:
print("type(path_trajectory1) =", type(path_trajectory1))
print("len(path_trajectory1) =", len(path_trajectory1) if path_trajectory1 is not None else None)

# Inspect the first few points
if path_trajectory1 is not None and len(path_trajectory1) > 0:
    print("first point =", path_trajectory1[0])
else:
    print("path_trajectory1 is EMPTY")

type(path_trajectory1) = <class 'list'>
len(path_trajectory1) = 30
first point = (327765.0612, 5815590.063)


In [19]:
print("p1[0] =", p1.iloc[0])

p1[0] = 1940;761;762;1089;1091;355;354;1096;342;343;1068;336;337;332;328;329;9;1104;627;1107;1075;1114;983;1110;1076;656;1108;1137;1143;2150


In [20]:
def UTM_to_lonlat(trajectory):
    
    location = []
    for i in range(len(trajectory)):
        utm_easting = trajectory[i][0]
        utm_northing = trajectory[i][1]
        lon, lat = transform(utm_zone, wgs84, utm_easting, utm_northing)
        
        location.append((lat,lon))
    return location

In [21]:
location1 = UTM_to_lonlat(path_trajectory1)
location2 = UTM_to_lonlat(path_trajectory2)
location3 = UTM_to_lonlat(path_trajectory3)

C:\Users\zhao\AppData\Local\Temp\ipykernel_63168\4230226251.py:7: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  lon, lat = transform(utm_zone, wgs84, utm_easting, utm_northing)


## 4. Map Visualization

Create an interactive Folium map with color-coded routes for each template.

In [22]:
location1

[(-37.790832988028704, 145.04390541230154),
 (-37.7906734575654, 145.03973061410068),
 (-37.790680721057804, 145.03644946946844),
 (-37.79082707344535, 145.03426900879913),
 (-37.79139339974763, 145.02833278441372),
 (-37.79157892111356, 145.02542068944516),
 (-37.791473566785946, 145.0213684205877),
 (-37.79118002574713, 145.01676611703613),
 (-37.79143041273408, 145.0144433219472),
 (-37.79199353155549, 145.01203209980218),
 (-37.79444845721229, 145.00495784432263),
 (-37.79557104195141, 145.00084072141436),
 (-37.795772664232494, 144.99903126811304),
 (-37.795781099026655, 144.9972917674624),
 (-37.79558837291697, 144.99496885537383),
 (-37.79516741931239, 144.99194802497098),
 (-37.79440215680436, 144.98993553487318),
 (-37.794210770759854, 144.98818057217326),
 (-37.79387004923743, 144.98509032755297),
 (-37.79328787500512, 144.97963534327195),
 (-37.793283597310655, 144.97929146202523),
 (-37.79298300678303, 144.97647182430717),
 (-37.79305193453652, 144.9756871293609),
 (-37.792

In [23]:
location1[0]

(-37.790832988028704, 145.04390541230154)

In [24]:
map = folium.Map(location=location1[0], zoom_start=12, tiles='cartodb positron')

# Define point coordinates as (lat, lon) tuples

# Create origin and destination markers
folium.Marker(location=location1[0], icon=folium.Icon(color='green', icon='info-sign')).add_to(map)
folium.Marker(location=location1[-1], icon=folium.Icon(color='red', icon='info-sign')).add_to(map)

# Draw directed polylines on the map using folium.PolyLine
folium.PolyLine(location1, color="#FA7F2C", weight=2, opacity=1).add_to(map)
folium.PolyLine(location2, color='blue', weight=2, opacity=1).add_to(map)
folium.PolyLine(location3, color='#00CCDD', weight=2, opacity=1).add_to(map)


# Save map to HTML file
map

In [25]:
map.save("map.html")

In [26]:
# 1) Save HTML first (optional but recommended)
map.save("map.html")

# 2) Export PNG (requires selenium + browser driver)
png_data = map._to_png(5)  # 5 seconds to wait for tile loading; adjust as needed

with open("map.png", "wb") as f:
    f.write(png_data)

print("Saved: map.html and map.png")

Saved: map.html and map.png


## 5. Data Export

Export node sequences and path data for further analysis.

In [27]:
node_sequence1 = 

SyntaxError: invalid syntax (1361266142.py, line 1)

In [ ]:
list_p1 = p1[0].split(';')

In [ ]:
'1935;4582;1843;1887;1861;1862;1891;1897;1849;1850;1844;1845;1901;2332'

In [ ]:
import numpy as np

# Original string
string_numbers = '1935;4582;1843;1887;1861;1862;1891;1897;1849;1850;1844;1845;1901;2332'

# Split string by semicolons to get a list of numbers
list_numbers = string_numbers.split(';')

# Convert each string number in the list to integer
int_numbers = [int(num) for num in list_numbers]

# Convert integer list to numpy array
numpy_array = np.array(int_numbers)

print(numpy_array)